# Household Electric Power Consumption — Forecasting with LSTM (from scratch) & Walk-Forward Evaluation
**Author:** Aytakin Mardani · aytakin57@gmail.com

Goal: forecast **next-day household electricity consumption (kWh/day)** from 4 years of minute-level data (UCI *Individual Household Electric Power Consumption*).

What this notebook shows:
1. Data preparation (daily aggregation, missing-data handling)
2. A **single-layer LSTM implemented from scratch in NumPy** (BPTT + Adam) and verified with a **numerical gradient check**
3. Chronological train / validation / test split (no leakage)
4. Comparison with baselines and classical ML (Ridge, Gradient Boosting)
5. Walk-forward results over 3 test years, statistical tests, prediction intervals

> Run `python src/prepare_data.py household_power_consumption.txt` once if you want to rebuild `data/daily_aggregated.csv` from the raw file.

In [ ]:
import sys, json, warnings; warnings.filterwarnings('ignore'); sys.path.insert(0, 'src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from common import load_daily, tabular, sequences, metrics
from lstm_np import LSTMRegressor, gradient_check

s = load_daily()                 # daily kWh, days with <50% coverage interpolated (max 14 days)
print(s.shape, s.index.min().date(), '->', s.index.max().date())
s.plot(figsize=(10, 3), lw=.8, title='Daily consumption (kWh/day)'); plt.show()

## 1. Is the hand-written LSTM correct? — numerical gradient check

In [ ]:
err = gradient_check()
print('max relative error between analytic and numerical gradients:', err)
assert err < 1e-6

## 2. Chronological split
Train: up to 2009-12-31 (last 90 days used for early stopping) · Test: 2010. Scaling statistics come from the training part only.

In [ ]:
X, C, y, idx = sequences(s, L=28)           # 28-day windows -> next day
train = idx <= '2009-12-31'; test = idx >= '2010-01-01'
mu, sd = y[train].mean(), y[train].std()
def scale(A): A = A.copy(); A[:, :, 0] = (A[:, :, 0] - mu) / sd; return A
Xtr, Ctr, ytr = X[train], C[train], y[train]; va = len(ytr) - 90
print(Xtr.shape, X[test].shape)

## 3. Train the LSTM (5 seeds, hidden=16, lr=0.003 — chosen on validation only)

In [ ]:
models = []
for seed in range(5):
    m = LSTMRegressor(n_in=6, n_cal=5, hidden=16, seed=seed)
    h = m.fit(scale(Xtr[:va]), Ctr[:va], (ytr[:va]-mu)/sd, scale(Xtr[va:]), Ctr[va:], (ytr[va:]-mu)/sd, epochs=80, lr=3e-3, seed=seed)
    models.append(m); print('seed', seed, 'epochs run:', len(h), 'best val MSE:', round(min(v for _, v in h), 3))
p_lstm = np.mean([m.predict(scale(X[test]), C[test]) for m in models], axis=0) * sd + mu
print('LSTM (5-seed mean):', metrics(y[test], p_lstm))

## 4. Baselines and classical ML on the same test days

In [ ]:
T = tabular(s); feats = [c for c in T.columns if c != 'y']; Ttr = T[T.index <= '2009-12-31']; Tte = T.loc[idx[test]]
p_naive = Tte['lag1'].values
p_ridge = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(Ttr[feats], Ttr['y']).predict(Tte[feats])
p_gb = HistGradientBoostingRegressor(max_iter=300, learning_rate=.05, max_depth=4, random_state=0).fit(Ttr[feats], Ttr['y']).predict(Tte[feats])
p_ens = (p_lstm + p_ridge + p_gb) / 3
rows = {n: metrics(y[test], p) for n, p in [('Naive (yesterday)', p_naive), ('Ridge', p_ridge), ('Gradient Boosting', p_gb), ('LSTM', p_lstm), ('Ensemble', p_ens)]}
pd.DataFrame(rows).T.round(3)

In [ ]:
plt.figure(figsize=(10, 3.4)); plt.plot(idx[test], y[test], color='grey', lw=1, label='actual'); plt.plot(idx[test], p_ens, color='C1', label='ensemble forecast')
plt.legend(); plt.ylabel('kWh/day'); plt.title('2010 test period'); plt.show()

## 5. Walk-forward results (3 test years), tests and prediction intervals
Produced by `python src/experiments.py` (≈1 minute on a laptop CPU).

In [ ]:
R = json.load(open('data/results_v2.json'))
print('Average MAE over 3 test years (kWh/day):')
print(pd.Series(R['avg_mae']).sort_values().round(2))
print('\nPaired Wilcoxon tests on 2010 absolute errors:')
print(pd.DataFrame(R['tests']).round(4))
print(f"\n90% conformal interval -> coverage {R['interval']['coverage']:.1f}%, half-width {R['interval']['q']:.1f} kWh")